# 02 · Honest baselines with classical machine learning

**Level:** beginner · **Time:** 40–55 minutes · **Data:** bundled digits · **Hardware:** CPU is sufficient.

You will split data before fitting, beat a trivial baseline, tune logistic regression using
validation data, and read accuracy, macro F1 and a confusion matrix.
**Prerequisite:** arrays and labels from notebook 01. This notebook creates its own data and models.

A classifier learns a mapping from features to class labels. A useful experiment must also
check performance on examples that did not influence fitting or model selection.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


## Step 1 — Set up the feature table
Each row contains 64 pixel intensities; each target is an integer 0–9.
Dividing by the dataset's known maximum is a fixed unit conversion, not a learned statistic.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             ConfusionMatrixDisplay)

digits = load_digits()
X = digits.data.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)
print("Features:", X.shape, "| targets:", y.shape)


## Step 2 — Reserve training, validation and test sets
We use about **70% training / 15% validation / 15% test**. Training fits parameters;
validation selects choices such as regularization strength; the test set estimates final
performance after those choices are fixed. `stratify` keeps class proportions similar.
Splitting row indices lets us verify that the sets do not overlap.

For this demonstration, random image splits are appropriate. For images grouped by person,
patient, video or acquisition session, split by that group to avoid near-duplicate leakage.


In [ ]:
all_indices = np.arange(len(y))
train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, stratify=y, random_state=42)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50,
    stratify=y[holdout_idx], random_state=42)

if SMOKE_TEST:
    # Fast verification mode preserves all ten classes.
    train_idx = train_test_split(train_idx, train_size=128,
                                stratify=y[train_idx], random_state=42)[0]
    val_idx = train_test_split(val_idx, train_size=64,
                              stratify=y[val_idx], random_state=42)[0]
    test_idx = train_test_split(test_idx, train_size=64,
                               stratify=y[test_idx], random_state=42)[0]

assert not (set(train_idx) & set(val_idx) or set(train_idx) & set(test_idx)
            or set(val_idx) & set(test_idx))
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]
print("Train / validation / test:", len(y_train), len(y_val), len(y_test))


## Step 3 — Inspect training data, then choose metrics
**Accuracy** is the fraction of correct predictions. **Macro F1** computes F1 separately
for each class, then weights each class equally. F1 balances precision (how trustworthy a
class prediction is) and recall (how many true members of a class we recover).
Accuracy alone can conceal weak performance on rare classes.


In [ ]:
counts = np.bincount(y_train, minlength=10)
plt.bar(np.arange(10), counts)
plt.xticks(np.arange(10))
plt.xlabel("Training class")
plt.ylabel("Number of examples")
plt.title("Training class balance")
plt.show()

def metric_row(name, target, predictions):
    return {"model": name, "accuracy": accuracy_score(target, predictions),
            "macro_f1": f1_score(target, predictions, average="macro", zero_division=0)}


## Step 4 — Start with a deliberately simple baseline
A `DummyClassifier` ignores the image and always predicts the most frequent training class.
Expect roughly 10% accuracy because the ten classes are fairly balanced.
A baseline checks whether a more elaborate model learns anything useful.


In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
baseline_result = metric_row("Most frequent class", y_val, dummy.predict(X_val))
pd.DataFrame([baseline_result])


## Step 5 — Put learned preprocessing inside a pipeline
Logistic regression learns a linear score for each class and converts scores into class probabilities.
Despite its name it is a **classification** method. `StandardScaler` learns one mean and
standard deviation per feature. The pipeline fits both stages using **training data only**,
then reuses those same statistics when predicting validation or test examples.

Fitting the scaler on all images before splitting would leak information about held-out data.
`C` controls inverse regularization strength: a smaller value penalizes large weights more.


In [ ]:
model = make_pipeline(StandardScaler(),
                      LogisticRegression(C=1.0, max_iter=2000, random_state=42))
model.fit(X_train, y_train)
val_predictions = model.predict(X_val)
initial_result = metric_row("Logistic regression, C=1", y_val, val_predictions)
print("Scaler fitted examples:", model.named_steps["standardscaler"].n_samples_seen_)
pd.DataFrame([baseline_result, initial_result]).round(4)


## Step 6 — Compare a small, declared set of choices on validation data
Try three values of `C`, choose the highest validation macro F1, and leave the test set alone.
In normal mode this small dataset often gives high accuracy, but your exact numbers may vary.
More validation trials can also overfit the validation set; record your choices.


In [ ]:
candidates = {}
search_rows = []
for strength in [0.1, 1.0, 10.0]:
    candidate = make_pipeline(
        StandardScaler(), LogisticRegression(C=strength, max_iter=2000, random_state=42))
    candidate.fit(X_train, y_train)
    candidates[strength] = candidate
    row = metric_row(f"C={strength}", y_val, candidate.predict(X_val))
    row["C"] = strength
    search_rows.append(row)

search_table = pd.DataFrame(search_rows).sort_values(
    ["macro_f1", "C"], ascending=[False, True])
best_C = float(search_table.iloc[0]["C"])
final_model = candidates[best_C]
print("Chosen C using validation only:", best_C)
search_table.round(4)


## Step 7 — Evaluate the fixed model once on the test set
The following predictions are the final evaluation for this experiment. We keep the chosen
model fitted on training only, so the reported protocol is simple and explicit.
Reusing these same stored predictions for a report, matrix and error gallery is one evaluation.
Do not choose new hyperparameters from the test report.


In [ ]:
test_predictions = final_model.predict(X_test)
final_metrics = metric_row(f"Final logistic regression C={best_C}", y_test, test_predictions)
print(pd.DataFrame([final_metrics]).round(4).to_string(index=False))
print(classification_report(y_test, test_predictions, labels=np.arange(10),
                            target_names=[str(i) for i in range(10)], zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=np.arange(10), cmap="Blues", colorbar=False)
plt.title("Final test confusion matrix: rows=true, columns=predicted")
plt.show()


## Step 8 — Inspect the mistakes
Off-diagonal matrix entries are errors. Look at a few to distinguish ambiguous handwriting,
preprocessing issues and systematic confusions. This is diagnosis; a new tuned experiment
needs validation decisions and, for an unbiased final claim, fresh held-out evaluation data.


In [ ]:
mistakes = np.flatnonzero(test_predictions != y_test)
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax in axes:
    ax.axis("off")
for ax, index in zip(axes, mistakes[:6]):
    ax.imshow(X_test[index].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"True {y_test[index]} / predicted {test_predictions[index]}")
plt.tight_layout()
plt.show()
print("Test mistakes:", len(mistakes), "out of", len(y_test))


## Practice 1 — Manually calculate baseline accuracy
Count how often the dummy prediction matches a validation label. Compare your fraction to
`baseline_result["accuracy"]`. Why is this model useful even though it is poor?

<details><summary>Sample answer</summary>

```python
guesses = dummy.predict(X_val)
manual_accuracy = float((guesses == y_val).sum() / len(y_val))
print(manual_accuracy, baseline_result["accuracy"])
```

It reveals how much performance we get without using the input features.
</details>


In [ ]:
guesses = dummy.predict(X_val)
print("First ten guesses:", guesses[:10])
# Calculate the fraction correct below.


## Practice 2 — Audit training-only scaling
Compare the scaler's stored means to `X_train.mean(axis=0)`. Why should we avoid
calling `fit_transform(X_val)`? Use `np.allclose` rather than exact float equality.

<details><summary>Sample answer</summary>

```python
scaler = final_model.named_steps["standardscaler"]
print(np.allclose(scaler.mean_, X_train.mean(axis=0), atol=1e-6))
scaled_val = scaler.transform(X_val)
print(scaled_val.shape)
```

Refitting on validation learns from held-out data and creates a different feature coordinate system.
</details>


In [ ]:
scaler = final_model.named_steps["standardscaler"]
print("Number of feature means:", len(scaler.mean_))
# Audit the means and transform validation inputs below.


## Practice 3 — Explain one validation confusion
Build a confusion matrix from **validation** predictions. Find the largest off-diagonal
entry and inspect examples from that pair. Keep this as a validation-only diagnostic.

<details><summary>Sample answer</summary>

```python
from sklearn.metrics import confusion_matrix
validation_guesses = final_model.predict(X_val)
cm = confusion_matrix(y_val, validation_guesses, labels=np.arange(10))
np.fill_diagonal(cm, 0)
true_class, predicted_class = np.unravel_index(cm.argmax(), cm.shape)
print(true_class, predicted_class, cm.max())
```

If all off-diagonal counts are zero, there is no validation confusion to inspect.
</details>


In [ ]:
validation_guesses = final_model.predict(X_val)
print("Validation errors to investigate:", int((validation_guesses != y_val).sum()))
# Build and inspect the matrix below; do not tune against the test results.


## What you learned
You established a baseline, preserved separate data roles, used a leakage-resistant pipeline,
selected regularization using validation macro F1 and inspected one final test evaluation.
A neural network must earn its complexity by improving something useful: accuracy,
robustness, deployment needs or the kinds of inputs it can represent.

**Checkpoint:** explain why a 99% training accuracy alone is not evidence of generalization.

**Next:** [03 · PyTorch tensors and automatic differentiation](03_pytorch_tensors_autograd.ipynb).

Primary references: [data splitting](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html),
[common pitfalls and leakage](https://scikit-learn.org/stable/common_pitfalls.html),
[logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html),
[classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics).
